# Scores Snapshot

Takes a weekly snapshot of `gold_person_scores` and `gold_branch_scores` into
two Delta tables. Designed to be run as a standalone Databricks job every Sunday
at 3am UTC (after the 2am OCR run).

**Idempotent**: if a snapshot for today already exists, the cell skips cleanly.

**Tables created**:
- `genealogy.gold_scores_snapshot_person` — one row per person per snapshot
- `genealogy.gold_scores_snapshot_branch` — one row per branch per snapshot

**Retention**: no automatic pruning — rows accumulate indefinitely.
At 8 branches + ~6,000 people per week this is ~300k person-rows/year, trivially small.

**Schedule**: Databricks Jobs UI → New Job → Notebook task → this notebook.
Cron: `0 0 3 * * 0` (Quartz syntax, Sunday 3am UTC).

## Cell 1 — Create tables if not exist

In [ ]:
# Create snapshot tables if they don't already exist.
# snapshot_date is the Sunday the job runs — used as the partition key.

spark.sql("""
CREATE TABLE IF NOT EXISTS genealogy.gold_scores_snapshot_person (
    snapshot_date       DATE,
    person_gedcom_id    STRING,
    branch              STRING,
    proximity_label     STRING,
    completeness_score  DOUBLE,
    evidence_score      DOUBLE,
    overall_score       DOUBLE,
    story_ready         BOOLEAN,
    story_written       BOOLEAN
)
USING DELTA
COMMENT 'Weekly snapshot of gold_person_scores for trend tracking'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS genealogy.gold_scores_snapshot_branch (
    snapshot_date         DATE,
    branch                STRING,
    avg_completeness      DOUBLE,
    avg_evidence          DOUBLE,
    avg_overall           DOUBLE,
    stories_written       INT,
    stories_ready         INT,
    direct_ancestors      INT,
    ancestor_fill_score   DOUBLE,
    depth_score           DOUBLE
)
USING DELTA
COMMENT 'Weekly snapshot of gold_branch_scores for trend tracking'
""")

print('Tables ready.')

## Cell 2 — Check for existing snapshot today (idempotency guard)

In [ ]:
from datetime import date

today = date.today().isoformat()  # e.g. '2026-03-22'

existing = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM genealogy.gold_scores_snapshot_branch
    WHERE snapshot_date = '{today}'
""").collect()[0]['n']

if existing > 0:
    print(f'Snapshot for {today} already exists ({existing} branch rows). Skipping.')
    dbutils.notebook.exit('already_done')
else:
    print(f'No snapshot for {today} found. Proceeding.')

## Cell 3 — Snapshot gold_scores_snapshot_person

In [ ]:
person_result = spark.sql(f"""
INSERT INTO genealogy.gold_scores_snapshot_person
SELECT
    DATE('{today}')      AS snapshot_date,
    person_gedcom_id,
    branch,
    proximity_label,
    ROUND(completeness_score, 4)  AS completeness_score,
    ROUND(evidence_score,     4)  AS evidence_score,
    ROUND(overall_score,      4)  AS overall_score,
    COALESCE(story_ready,   FALSE) AS story_ready,
    COALESCE(story_written, FALSE) AS story_written
FROM genealogy.gold_person_scores
""")

person_count = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM genealogy.gold_scores_snapshot_person
    WHERE snapshot_date = '{today}'
""").collect()[0]['n']

print(f'Person snapshot done: {person_count} rows written for {today}.')

## Cell 4 — Snapshot gold_scores_snapshot_branch

In [ ]:
# gold_branch_scores already has the aggregated columns we need.
# direct_ancestors: count of people with ancestral_proximity = 0 in this branch.
# ancestor_fill_score: direct ancestors found at gen 3-8 as % of 63 per-branch slots.
# depth_score: max generation depth as % of 21-generation ceiling.
#
# Note: gold_generation_depth.person_id holds the same values as
# gold_person_branch.person_gedcom_id — column rename is pending.

branch_result = spark.sql(f"""
INSERT INTO genealogy.gold_scores_snapshot_branch
WITH tree_shape AS (
    SELECT
        gpb.branch,
        COUNT(DISTINCT CASE WHEN ggd.generation_depth BETWEEN 3 AND 8
                            THEN ggd.person_id END)          AS ancestors_found,
        MAX(ggd.generation_depth)                             AS max_depth
    FROM genealogy.gold_generation_depth ggd
    JOIN genealogy.gold_person_branch gpb
        ON gpb.person_gedcom_id = ggd.person_id
    WHERE ggd.generation_depth >= 3
    GROUP BY gpb.branch
)
SELECT
    DATE('{today}')                            AS snapshot_date,
    bs.branch,
    ROUND(bs.avg_completeness, 4)              AS avg_completeness,
    ROUND(bs.avg_evidence,     4)              AS avg_evidence,
    ROUND(bs.avg_overall,      4)              AS avg_overall,
    COALESCE(bs.stories_written, 0)            AS stories_written,
    COALESCE(bs.stories_ready,   0)            AS stories_ready,
    COALESCE(da.direct_ancestors, 0)           AS direct_ancestors,
    ROUND(COALESCE(ts.ancestors_found, 0)
          / 63.0 * 100, 4)                    AS ancestor_fill_score,
    ROUND(COALESCE(ts.max_depth, 0)
          / 21.0 * 100, 4)                    AS depth_score
FROM genealogy.gold_branch_scores bs
LEFT JOIN (
    SELECT branch, COUNT(*) AS direct_ancestors
    FROM genealogy.gold_person_scores
    WHERE ancestral_proximity = 0
    GROUP BY branch
) da ON da.branch = bs.branch
LEFT JOIN tree_shape ts ON ts.branch = bs.branch
""")

branch_count = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM genealogy.gold_scores_snapshot_branch
    WHERE snapshot_date = '{today}'
""").collect()[0]['n']

print(f'Branch snapshot done: {branch_count} rows written for {today}.')

## Cell 5 — Verify: show latest two snapshots

In [ ]:
display(spark.sql("""
    SELECT snapshot_date, branch, avg_completeness, avg_evidence, avg_overall,
           stories_written, stories_ready,
           ancestor_fill_score, depth_score
    FROM genealogy.gold_scores_snapshot_branch
    WHERE snapshot_date IN (
        SELECT DISTINCT snapshot_date
        FROM genealogy.gold_scores_snapshot_branch
        ORDER BY snapshot_date DESC
        LIMIT 2
    )
    ORDER BY snapshot_date DESC, branch
"""))